# WeatherGPT events: implementation notebook

**What this is.** Twenty small per-target models (LightGBM) that post-process the newest NOAA **GFS** and **GEFS** run into calibrated answers for any
*land* point in India (6-38 N, 67-98 E) up to **10 days** ahead: thunderstorm / fog / strong-wind / rain chances, rain *amount*, temperature,
wind and humidity *ranges*, heat-/cold-wave days and 3-/7-day rain windows. An orchestrator LLM is meant to call only the models a question needs.

**What this notebook does, end to end:** load the published artifacts -> look at the admission gate -> fetch the live run ->
forecast a place -> plot -> compare cities -> hand a tool call to the validated dispatcher -> convert answers to the app's evidence objects -> call the HTTP API.

**Honesty first.** A model answers only where its own held-out metrics prove skill (unseen 4x4 degree blocks, future dates) and otherwise returns
*no numbers* and says why. Research prototype; not an official forecast.

Needs internet (Hugging Face for the artifacts, NOAA AWS for the live run, about 1.5 minutes the first time). Works on Kaggle or Colab (CPU is enough).

## 1. Setup

In [ ]:
%pip install -q lightgbm eccodes huggingface_hub matplotlib pandas scipy httpx pydantic fastapi

In [ ]:
import os, sys, subprocess, pathlib

SOURCE = os.environ.get("WEATHERGPT_EVENTS_SOURCE", "Arko007/weathergpt-events")   # Hugging Face repo id, or a local artifact folder
HF_TOKEN = os.environ.get("HF_TOKEN")                                             # only needed while the repo is private
CODE_REPO = "https://github.com/Anamitra-Sarkar/weathergpt"                       # inference + feature code (single source of truth)

try:
    import weathergpt_events                                                       # already importable (running inside the repo)?
except ImportError:
    if not pathlib.Path("weathergpt").exists():
        subprocess.run(["git", "clone", "--depth", "1", CODE_REPO, "weathergpt"], check=True)
    sys.path.insert(0, str(pathlib.Path("weathergpt").resolve()))
    import weathergpt_events
print("code from:", pathlib.Path(weathergpt_events.__file__).parent)

## 2. Load the artifacts and look at the admission gate
`load_engine` reads every model's `metrics.json` and **loads only the ones that pass the gate** (provenance, skill over zone x month climatology *and* the global base rate on places never seen). The rest are listed with the exact reason.

In [ ]:
import pandas as pd
from weathergpt_events.loader import load_engine
from weathergpt_events.service import ForecastService, summarise, ToolCallError

engine = load_engine(SOURCE, token=HF_TOKEN)
status = pd.DataFrame(engine.registry.status())
print(f"{int(status['loaded'].sum())} of {len(status)} artifacts served")
status[["target", "loaded", "reason"]].assign(reason=lambda d: d["reason"].str.slice(0, 110))

## 3. The tool catalogue an orchestrator LLM sees
Built from the loaded registry, not typed by hand: it can only advertise models that passed the gate, with the climate zones their own held-out metrics support.

In [ ]:
import json
service_stub = ForecastService(engine, run_provider=lambda: None)          # catalogue needs no live run
catalogue = service_stub.catalogue()
table = pd.DataFrame([{"tool": t["name"], "kind": t["kind"], "horizon_days": t["horizon_days"], "zones_validated": len(t["validated_zones"]),
                       "answers": t["answers"][:80]} for t in catalogue if t["available"]])
display(table)
print(json.dumps({k: catalogue[0][k] for k in ("name", "answers", "parameters")}, indent=1)[:900])

## 4. Skill of every served model (copied from each model's own `metrics.json`)
Left: probability models, Brier skill score over the *zone x month climatology* on unseen places. Right: range models, median error vs the raw GFS value (lower is better).

In [ ]:
import matplotlib.pyplot as plt

bin_rows = [(t["name"], t["headline_held_out_skill"]["bss_vs_zone_month"]) for t in catalogue
            if t["available"] and t["kind"] in ("binary", "xent")]
rng_rows = [(t["name"], t["headline_held_out_skill"]["median_mae"], t["headline_held_out_skill"]["gfs_raw_mae"]) for t in catalogue
            if t["available"] and t["kind"] == "quantile"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
names, vals = zip(*sorted(bin_rows, key=lambda r: r[1]))
ax[0].barh(names, vals, color="#2a6f97"); ax[0].set_xlabel("Brier skill score vs zone x month climatology (unseen places)"); ax[0].axvline(0, color="k", lw=.6)
x = range(len(rng_rows)); w = .38
ax[1].bar([i - w / 2 for i in x], [r[1] for r in rng_rows], w, label="WeatherGPT median", color="#2a6f97")
ax[1].bar([i + w / 2 for i in x], [r[2] for r in rng_rows], w, label="raw GFS", color="#bbbbbb")
ax[1].set_xticks(list(x)); ax[1].set_xticklabels([r[0] for r in rng_rows], rotation=25, ha="right"); ax[1].set_ylabel("mean absolute error (deg C, m/s or %)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Fetch the newest live GFS + GEFS run
`ForecastService.live` checks which 00Z run is complete on NOAA's AWS archive, downloads the India window (52 forecast steps, byte-range + GRIB decode) and caches it. If the newest run is older than the freshness limit it **refuses** instead of serving a stale forecast.

In [ ]:
import time
t0 = time.time()
service = ForecastService.live(engine, cache_dir="run_cache")
run = service.current_run()
print("newest fresh run:", None if run is None else run.run_date, f"| {time.time() - t0:.0f}s")
assert run is not None, "no fresh run published right now - try again later"

## 6. Forecast a place
Delhi, all served models, 10 days. Each entry says whether it is `validated` for the place's climate zone; entries that are not carry **no numbers**.

In [ ]:
PLACES = {"Delhi": (28.61, 77.21), "Mumbai": (19.08, 72.88), "Chennai": (13.08, 80.27), "Jaisalmer": (26.92, 70.92),
          "Cherrapunji": (25.30, 91.70), "Leh": (34.15, 77.58), "Bengaluru": (12.97, 77.59), "Kolkata": (22.57, 88.36)}
lat, lon = PLACES["Delhi"]
delhi = service.forecast(lat, lon, horizon_days=10)
print(summarise(delhi, max_rows=2))

## 7. Plots for one place
Rain chance per day (and the amount *if* it rains), the daily-maximum range, the 3-hourly thunderstorm chance, and the 7-day rain curve.

In [ ]:
def entries(result, target):
    res = result["targets"].get(target, {})
    return [e for e in res.get("entries", []) if e.get("validated")] if res.get("available") else []

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
rain = entries(delhi, "rain_curve")
if rain:
    days = [e["utc_day"][5:] for e in rain]
    ax[0, 0].bar(days, [e["p_any_rain"] for e in rain], color="#8ecae6", label="P(rain >= 1 mm)")
    ax[0, 0].plot(days, [e["p_rainy_day"] for e in rain], "o-", color="#023047", label="P(rain >= 2.5 mm)  (IMD 'rainy day')")
    ax[0, 0].set_ylim(0, 1); ax[0, 0].set_title("Daily rain chance"); ax[0, 0].legend(); ax[0, 0].tick_params(axis="x", rotation=45)
tmax = entries(delhi, "tmax_range")
if tmax:
    d = [e["ist_day"][5:] for e in tmax]
    ax[0, 1].fill_between(d, [e["lo"] for e in tmax], [e["hi"] for e in tmax], color="#ffd6a5", label="80% range (conformal)")
    ax[0, 1].plot(d, [e["q50"] for e in tmax], "o-", color="#d00000", label="median"); ax[0, 1].set_title("Daily maximum temperature (deg C)")
    ax[0, 1].legend(); ax[0, 1].tick_params(axis="x", rotation=45)
storm = entries(delhi, "thunderstorm")
if storm:
    ax[1, 0].plot(pd.to_datetime([e["valid_time_utc"] for e in storm]), [e["probability"] for e in storm], color="#6a4c93")
    ax[1, 0].set_title("Thunderstorm chance in each 3 h window"); ax[1, 0].tick_params(axis="x", rotation=30)
week = entries(delhi, "rain_7day_total")
if week:
    ex = week[0]["exceedance"]
    ax[1, 1].step([float(k) for k in ex], list(ex.values()), where="post", color="#386641")
    ax[1, 1].set_xscale("symlog", linthresh=5); ax[1, 1].set_xlabel("7-day rain total t (mm)"); ax[1, 1].set_ylabel("P(total >= t)")
    ax[1, 1].set_title(f"7-day rain total from {week[0]['window_start_utc_day']}")
plt.tight_layout(); plt.show()

## 8. Compare cities
Three-day rain chance, tomorrow's maximum temperature, and whether the models are validated for that city's climate zone.

In [ ]:
rows = []
for name, (la, lo) in PLACES.items():
    r = service.forecast(la, lo, targets=["rain_3day_any_2p5mm", "tmax_range"], horizon_days=3)
    if not r["available"]:
        rows.append({"place": name, "note": r["reason"][:60]}); continue
    rain3, tm = entries(r, "rain_3day_any_2p5mm"), entries(r, "tmax_range")
    rows.append({"place": name, "zone": r["targets"]["tmax_range"].get("zone"), "coastal_cell": r["location"]["coastal"],
                 "P(>=2.5 mm in next 3 days)": rain3[0]["probability"] if rain3 else None,
                 "tomorrow Tmax median": tm[0]["q50"] if tm else None, "Tmax 80% range": f"{tm[0]['lo']:.1f}-{tm[0]['hi']:.1f}" if tm else "not validated here"})
pd.DataFrame(rows).round(2)

## 9. Refusals are part of the product
A point over the sea, outside India, or a model without skill in the place's zone is answered with a reason, never with a guess.

In [ ]:
for label, (la, lo) in {"Arabian Sea": (15.0, 68.5), "London": (51.5, -0.1), "Puri (coastal, 24% land in the GFS cell)": (19.81, 85.83)}.items():
    r = service.forecast(la, lo, targets=["thunderstorm"], horizon_days=1)
    print(f"{label:45s} ->", "answered" if r["available"] else r["reason"])
print()
unvalidated = [t for t, res in delhi["targets"].items() if res.get("available") and not any(e["validated"] for e in res["entries"])]
print("targets with no validated answer for Delhi's zone:", unvalidated or "none")

## 10. The orchestrator interface
An LLM planner emits a tool call; the service validates it against the catalogue (unknown tool, bad or extra arguments, impossible horizon are rejected *before* anything runs) and executes it.

In [ ]:
call = {"name": "rain_curve", "arguments": {"lat": 19.08, "lon": 72.88, "horizon_days": 3}}        # what the planner would emit for "will it rain in Mumbai this week-end?"
answer = service.call_tool(call["name"], call["arguments"])
print(summarise(answer, max_rows=3))
for bad in ({"name": "make_it_rain", "arguments": {}}, {"name": "rain_curve", "arguments": {"lat": 19.08, "lon": 72.88, "horizon_days": 30}},
            {"name": "rain_curve", "arguments": {"lat": "Mumbai", "lon": 72.88}}):
    try:
        service.call_tool(bad["name"], bad["arguments"])
    except ToolCallError as exc:
        print("rejected:", exc)

## 11. Evidence objects for the WeatherGPT backend
Only validated entries become Canonical Evidence Objects, each keeping its own variable and statistic so the semantic gate can refuse to mix them (a probability stays a probability; a rain amount is flagged *conditional on rain*).

In [ ]:
try:
    from weathergpt_events.ceo_bridge import to_ceos
    ceos = to_ceos(service.forecast(19.08, 72.88, targets=["rain_curve", "tmax_range"], horizon_days=3))
    display(pd.DataFrame([{"variable": c.variable.value, "statistic": c.statistic.value, "value": c.value, "probability": c.probability, "unit": c.unit,
                           "valid_from": c.valid_from.strftime("%m-%d %H:%M"), "conditional": c.extra.get("conditional_on_wet", False)} for c in ceos]).head(12))
except ImportError as exc:
    print("evidence bridge needs the full repo (app/ package):", exc)

## 12. As an HTTP service
The same service behind a FastAPI router (mount it in the backend with `app.include_router(create_router(service), prefix='/events')`).

In [ ]:
from fastapi.testclient import TestClient
from weathergpt_events.api import create_app

client = TestClient(create_app(service))
print(client.get("/status").json()["run"], "| served:", len(client.get("/status").json()["served"]))
r = client.get("/forecast", params={"lat": 22.57, "lon": 88.36, "targets": "rain_curve,tmax_range", "horizon_days": 2})
print(r.status_code, summarise(r.json(), max_rows=2))
print(client.post("/tool", json={"name": "nope", "arguments": {}}).json())

## 13. Limits to keep in mind
* Skill is measured on unseen places and future dates, in a **single run (one seed, no confidence intervals)**. Treat differences of a few thousandths as noise.
* Rain skill is measured against **CHIRPS**, a satellite-gauge product that itself agrees with METAR rain reports only moderately.
* Daily rain beyond about 90 mm has no demonstrated skill over climatology; the curve flags those thresholds (`thresholds_without_skill_mm`).
* Heat-/cold-wave labels approximate the IMD criteria from METAR station normals (not IMD declarations); `coldwave_imd` is refused by the gate.
* 00Z cycle only; no nowcast from live observations; places with little land in the 0.25 degree GFS cell (e.g. Puri) are refused.
* More: `docs/EVENT_MODELS_GUIDE.md` and `docs/AGENTIC_ARCHITECTURE.md` in the repo.